# TTRPG Trends Analysis

Visual exploration of objective TTRPG statistics: crowdfunding funding trends, platform shifts, system popularity, and market/player indicators.

**Data:** CSV files in `../data/` (see `../data/README.md` for source provenance).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_DIR = Path("../data")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

def load(name: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / name)

datasets = sorted(p.name for p in DATA_DIR.glob("*.csv"))
print(f"Found {len(datasets)} datasets in {DATA_DIR.resolve()}")
for name in datasets:
    print(f"  - {name}")

## 1. RPG Crowdfunding Volume (2013–2025)

Source: Skalchemist all-platform RPG crowdfunding report. Values inflation-adjusted to Nov 2025 USD.

In [ ]:
yearly = load("kickstarter_rpg_crowdfunding_yearly.csv")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total funding by year
ax = axes[0]
bars = ax.bar(yearly["year"], yearly["total_value_usd"] / 1e6, color="#4C72B0", edgecolor="white")
ax.set_title("Total RPG Crowdfunding Funding")
ax.set_xlabel("Year")
ax.set_ylabel("USD (millions)")
ax.set_xticks(yearly["year"][::2])
for bar, val in zip(bars, yearly["total_value_usd"]):
    if val > 50e6:
        ax.annotate(f"${val/1e6:.0f}M", xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=9)

# Project count vs mean value
ax = axes[1]
ax2 = ax.twinx()
ax.plot(yearly["year"], yearly["project_count"], "o-", color="#55A868", label="Projects")
ax2.plot(yearly["year"], yearly["mean_project_value_usd"] / 1e3, "s-", color="#C44E52", label="Mean value ($K)")
ax.set_title("More Projects, Lower Average Funding (2025)")
ax.set_xlabel("Year")
ax.set_ylabel("Project count", color="#55A868")
ax2.set_ylabel("Mean project value ($K)", color="#C44E52")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

yearly.tail(3)

## 2. Kickstarter Tabletop Games Success Rate

Source: Kickstarter Creative Download 2024 / RPG Drop analysis.

In [ ]:
success = load("kickstarter_success_rate_yearly.csv")
ks_2024 = load("kickstarter_tabletop_games_2024.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(success["year"], success["success_rate_pct"], "o-", linewidth=2.5, markersize=10, color="#4C72B0")
ax.fill_between(success["year"], success["success_rate_pct"], alpha=0.15, color="#4C72B0")
ax.set_title("Kickstarter Tabletop Success Rate")
ax.set_xlabel("Year")
ax.set_ylabel("Success rate (%)")
ax.set_ylim(65, 85)
for _, row in success.iterrows():
    ax.annotate(f"{row['success_rate_pct']:.0f}%", (row["year"], row["success_rate_pct"]),
                textcoords="offset points", xytext=(0, 10), ha="center")

# 2024 tabletop snapshot
ax = axes[1]
metrics = ks_2024.set_index("metric")
labels = ["Funded projects", "Launched projects", "Million-$ campaigns"]
values = [
    metrics.loc["tabletop_projects_funded", "value"],
    metrics.loc["tabletop_projects_launched", "value"],
    metrics.loc["million_dollar_games_campaigns", "value"],
]
colors = ["#55A868", "#4C72B0", "#DD8452"]
ax.barh(labels, values, color=colors)
ax.set_title("Kickstarter Tabletop Games — 2024")
ax.set_xlabel("Count")
for i, v in enumerate(values):
    ax.text(v + 50, i, f"{int(v):,}", va="center")

plt.tight_layout()
plt.show()

## 3. Crowdfunding Platform Share (2025)

Kickstarter still hosts most projects, but Backerkit captures a growing share of funding.

In [ ]:
platforms = load("crowdfunding_platform_share_2025.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = sns.color_palette("Set2", len(platforms))

for ax, col, title in zip(axes, ["project_share_pct", "funding_share_pct"],
                          ["Share of Projects", "Share of Funding"]):
    wedges, texts, autotexts = ax.pie(
        platforms[col], labels=platforms["platform"], autopct="%1.0f%%",
        colors=colors, startangle=90, pctdistance=0.75
    )
    ax.set_title(f"RPG Crowdfunding Platform — {title} (2025)")

plt.tight_layout()
plt.show()

## 4. Non-D&D 5E System Popularity on Crowdfunding (2025)

Systems with 5+ related crowdfunding projects. Shadowdark and OSR-adjacent games lead outside D&D 5E supplements.

In [ ]:
systems = load("crowdfunding_non5e_systems_2025.csv").sort_values("project_count", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(systems["system"], systems["project_count"], color=sns.color_palette("viridis", len(systems)))
ax.set_title("Non-D&D 5E Systems — Crowdfunding Project Count (2025)")
ax.set_xlabel("Number of related projects")
for i, (count, name) in enumerate(zip(systems["project_count"], systems["system"])):
    ax.text(count + 1, i, str(count), va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5. November Crowdfunding Year-over-Year

Comparing successful November campaigns on Kickstarter and Backerkit (2023–2025).

In [ ]:
nov = load("crowdfunding_november_yoy.csv")

# Pivot campaign counts by platform
count_pivot = nov[nov["metric"] == "campaign_count"].pivot(
    index="year", columns="platform", values="value_usd"
)
pledge_pivot = nov[nov["metric"] == "total_pledged"].pivot(
    index="year", columns="platform", values="value_usd"
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Campaign counts
count_pivot.plot(kind="bar", ax=axes[0, 0], color=["#4C72B0", "#DD8452"])
axes[0, 0].set_title("November Campaign Count by Platform")
axes[0, 0].set_ylabel("Campaigns")
axes[0, 0].tick_params(axis="x", rotation=0)
axes[0, 0].legend(title="Platform")

# Total pledged
(pledge_pivot / 1e6).plot(kind="bar", ax=axes[0, 1], color=["#4C72B0", "#DD8452"])
axes[0, 1].set_title("November Total Pledged by Platform")
axes[0, 1].set_ylabel("USD (millions)")
axes[0, 1].tick_params(axis="x", rotation=0)
axes[0, 1].legend(title="Platform")

# D&D 5E trend
dnd = nov[(nov["platform"] == "D&D_5E") & (nov["metric"].isin(["campaign_count", "total_pledged"]))]
dnd_wide = dnd.pivot(index="year", columns="metric", values="value_usd")
ax = axes[1, 0]
ax.bar(dnd_wide.index - 0.2, dnd_wide["campaign_count"], width=0.4, label="Campaigns", color="#8172B3")
ax2 = ax.twinx()
ax2.plot(dnd_wide.index, dnd_wide["total_pledged"] / 1e6, "o-", color="#C44E52", label="Pledged ($M)")
ax.set_title("D&D 5E November Crowdfunding")
ax.set_xlabel("Year")
ax.set_ylabel("Campaign count")
ax2.set_ylabel("Pledged (USD millions)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

# AI-assisted campaigns
ai = nov[(nov["platform"] == "AI_assisted") & (nov["metric"].isin(["campaign_count", "total_pledged"]))]
ai_wide = ai.pivot(index="year", columns="metric", values="value_usd")
ax = axes[1, 1]
ax.bar(ai_wide.index, ai_wide["campaign_count"], color="#CCB974", label="Campaign count")
ax.set_title("AI-Assisted TTRPG Campaigns (November)")
ax.set_xlabel("Year")
ax.set_ylabel("Campaign count")
ax2 = ax.twinx()
ax2.plot(ai_wide.index, ai_wide["total_pledged"] / 1e3, "s--", color="#8C564B", label="Pledged ($K)")
ax2.set_ylabel("Total pledged ($K)")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

## 6. Top-Funded TTRPG Kickstarters (2024)

Source: Kickstarter / RPG Drop Creative Download 2024.

In [ ]:
top_2024 = load("kickstarter_top_ttrpg_2024.csv").sort_values("total_raised_usd", ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_2024["project"], top_2024["total_raised_usd"] / 1e6, color="#4C72B0")
ax.set_title("Top-Funded TTRPG Kickstarter Campaigns (2024)")
ax.set_xlabel("Total raised (USD millions)")
for i, (proj, raised, backers) in enumerate(
    zip(top_2024["project"], top_2024["total_raised_usd"], top_2024["backers"])
):
    ax.text(raised / 1e6 + 0.1, i, f"${raised/1e6:.1f}M · {int(backers):,} backers", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Market & Player Indicators

Global market estimates, player-base growth, genre share, and official D&D engagement metrics.

In [ ]:
market = load("market_indicators.csv")
engagement = load("player_engagement_wotc_2024.csv")
rankings = load("hobby_store_rpg_rankings.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Player base growth
players = market[market["indicator"].str.contains("active_players")]
ax = axes[0, 0]
bars = ax.bar(players["year"].astype(str), players["value"], color=["#A1C9F4", "#FFB482"])
ax.set_title("Estimated Active TTRPG Players Worldwide")
ax.set_ylabel("Millions of players")
for bar, row in zip(bars, players.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{int(row.value)}M", ha="center")

# Genre & purchase channel
genre_pct = market[market["indicator"].isin([
    "medieval_fantasy_genre_share", "rpg_purchases_digital_pct", "players_18_35_play_quarterly_pct"
])]
ax = axes[0, 1]
labels = ["Fantasy genre share", "Digital purchases", "18–35 play quarterly"]
ax.barh(labels, genre_pct["value"].values, color=["#8DE5A1", "#FF9F9B", "#D0BBFF"])
ax.set_title("Player Preference Indicators (2024)")
ax.set_xlabel("Percent")
for i, v in enumerate(genre_pct["value"].values):
    ax.text(v + 1, i, f"{v:.0f}%", va="center")

# D&D Beyond engagement
ax = axes[1, 0]
eng = engagement[engagement["metric"] != "great_ttrpg_survey_responses"].copy()
eng["label"] = eng["metric"].str.replace("dnd_", "").str.replace("_", " ").str.title()
eng["value_m"] = eng.apply(
    lambda r: r["value"] / 1e6 if r["value"] > 1e4 else r["value"] / 1e3, axis=1
)
eng["display"] = eng.apply(
    lambda r: f"{r['value_m']:.1f}M" if r["value"] > 1e4 else f"{int(r['value']):,}", axis=1
)
ax.barh(eng["label"], eng["value_m"], color="#4C72B0")
ax.set_title("D&D Engagement Metrics (WotC, 2024)")
ax.set_xlabel("Value (millions where applicable)")
for i, (v, d) in enumerate(zip(eng["value_m"], eng["display"])):
    ax.text(v + 0.3, i, d, va="center", fontsize=9)

# Hobby store rankings (rank order — ICv2 does not publish dollar share)
ax = axes[1, 1]
rank_data = rankings.drop_duplicates("rpg_line").sort_values("rank")
ax.barh(rank_data["rpg_line"][::-1], rank_data["rank"][::-1], color=sns.color_palette("rocket", len(rank_data)))
ax.set_title("ICv2 Hobby Store RPG Rankings (2024)")
ax.set_xlabel("Rank (lower = higher sales)")
ax.invert_xaxis()
ax.text(0.95, 0.05, "Rank order only — ICv2 does not publish % share",
        transform=ax.transAxes, ha="right", fontsize=9, style="italic", color="gray")

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Crowdfunding volume is up, average project value is down** — 2,331 RPG projects in 2025 (record), but mean funding fell to $28.7K (lowest since 2014).
2. **Kickstarter tabletop success rate hit 80% in 2024** — the highest in Kickstarter's 15-year history, even as total pledged dollars cooled in 2025.
3. **Platform diversification** — Backerkit grew to 13% of projects and 25% of funding by 2025.
4. **Beyond D&D 5E** — Shadowdark, Mothership, and Old School Essentials lead non-5E crowdfunding; Pathfinder 2E remains prominent.
5. **D&D still dominates retail** — ICv2 hobby-channel rankings place D&D first; industry estimates put D&D at ~50% of hobby-store RPG sales by dollar value.
6. **Player base is growing** — Estimates rose from 27M active players (2018) to 42M (2024); fantasy remains the dominant genre (~54% market share).